In [19]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("GPU naam:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "geen GPU")

CUDA beschikbaar: True
GPU naam: Tesla T4


In [20]:
!pip install -q transformers

In [21]:
import pandas as pd

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)

print("Aantal rijen:", len(df))
print("Kolommen:", df.columns.tolist())
df.head()

Aantal rijen: 2050
Kolommen: ['num', 'name', 'p_np', 'smiles']


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


In [22]:
print(df["p_np"].value_counts())
print("\nFractie positief:", df["p_np"].mean().round(3))

p_np
1    1567
0     483
Name: count, dtype: int64

Fractie positief: 0.764


In [23]:
print("Missende SMILES:", df["smiles"].isna().sum())
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

Missende SMILES: 0


In [24]:
from sklearn.model_selection import train_test_split

smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

# Eerst test eraf halen (10%)
train_smiles, test_smiles, train_labels, test_labels = train_test_split(
    smiles, labels, test_size=0.1, random_state=42, stratify=labels
)
# Daarna val van train (10% van de rest)
train_smiles, val_smiles, train_labels, val_labels = train_test_split(
    train_smiles, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_smiles)} | Val: {len(val_smiles)} | Test: {len(test_smiles)}")

Train: 1660 | Val: 185 | Test: 205


In [25]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "DeepChem/ChemBERTa-77M-MLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
sample = "CC(=O)Oc1ccccc1C(=O)O"  # aspirine
tokens = tokenizer.tokenize(sample)
print("SMILES:", sample)
print("Tokens:", tokens)
print("Aantal tokens:", len(tokens))


SMILES: CC(=O)Oc1ccccc1C(=O)O
Tokens: ['C', 'C', '(', '=', 'O', ')', 'O', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', 'C', '(', '=', 'O', ')', 'O']
Aantal tokens: 21


In [27]:
import torch
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer, max_length=128):
        self.smiles = smiles
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SMILESDataset(train_smiles, train_labels, tokenizer)
val_ds   = SMILESDataset(val_smiles,   val_labels,   tokenizer)
test_ds  = SMILESDataset(test_smiles,  test_labels,  tokenizer)

print("Aantal samples in train_ds:", len(train_ds))
print("Voorbeeld input_ids shape:", train_ds[0]["input_ids"].shape)

Aantal samples in train_ds: 1660
Voorbeeld input_ids shape: torch.Size([128])


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }

In [29]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./chemberta_bbbp",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=True,            # mixed-precision: sneller op T4 GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Roc Auc
1,0.684521,0.612982,0.767568,0.881286
2,0.518419,0.480428,0.762162,0.919568
3,0.470776,0.413199,0.789189,0.936251
4,0.402467,0.362251,0.864865,0.943746
5,0.350020,0.323620,0.864865,0.947614
6,0.326019,0.299315,0.891892,0.949387
7,0.324869,0.284652,0.902703,0.951161
8,0.290567,0.274836,0.908108,0.950838
9,0.288878,0.269837,0.908108,0.951161
10,0.273990,0.268252,0.908108,0.951322


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.beta', 'roberta.embeddings.LayerNorm.gamma', 'roberta.encoder.layer.0.attention.output.LayerNorm.beta', 'roberta.enc

TrainOutput(global_step=520, training_loss=0.3947388062110314, metrics={'train_runtime': 21.3824, 'train_samples_per_second': 776.338, 'train_steps_per_second': 24.319, 'total_flos': 38242141900800.0, 'train_loss': 0.3947388062110314, 'epoch': 10.0})

In [31]:
test_metrics = trainer.evaluate(test_ds)

print("=== Testresultaten ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}")

=== Testresultaten ===
eval_loss                : 0.2404
eval_accuracy            : 0.9073
eval_roc_auc             : 0.9686
eval_runtime             : 0.1066
eval_samples_per_second  : 1922.7280
eval_steps_per_second    : 37.5170
epoch                    : 10.0000


In [32]:
from sklearn.metrics import confusion_matrix, classification_report

# Voorspellingen op de test-set
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print("                 voorspeld 0   voorspeld 1")
print(f"  werkelijk 0:    {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  werkelijk 1:    {cm[1,0]:5d}        {cm[1,1]:5d}")

print("\nPer-klasse metrics:")
print(classification_report(y_true, y_pred, target_names=["geen BBB (0)", "wel BBB (1)"]))

Confusion matrix:
                 voorspeld 0   voorspeld 1
  werkelijk 0:       41            7
  werkelijk 1:       12          145

Per-klasse metrics:
              precision    recall  f1-score   support

geen BBB (0)       0.77      0.85      0.81        48
 wel BBB (1)       0.95      0.92      0.94       157

    accuracy                           0.91       205
   macro avg       0.86      0.89      0.88       205
weighted avg       0.91      0.91      0.91       205

